In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import pandas as pd
from jppype import Mosaic, vscode_theme
from tqdm import tqdm

from fundus_toolkits import FundusData
from fundus_toolkits.utils.data_io import most_common_image_ext
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset, SampleInfo

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [3]:
DATASETS_ROOT = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/")
DATASETS_PATH = {
    dataset: DATASETS_ROOT / folder
    for dataset, folder in {
        "GAVE-train": "GAVE-train",
        "MAPLES-DR": "MAPLES-DR",
        "FundusAV": "Fundus-AV",
        "HRF": "HRF",
        "LES-AV": "LES-AV",
        "INSPIRE": "INSPIRE",
        "DRIVE_train": "AV_DRIVE/training",
        "DRIVE_test": "AV_DRIVE/test",
    }.items()
}

## Preprocess Datasets


In [4]:
from fundus_vessels_toolkit.models.wrappers.automorph import automorph_segment_av
from fundus_vessels_toolkit.models.wrappers.vascx import vascx_segment_av

from fundus_vessels_toolkit.models import segment_av


for dataset_path in DATASETS_PATH.values():
    raw_path = dataset_path / "1-images"
    for raw_file in tqdm(Path(raw_path).glob(f"*{most_common_image_ext(raw_path)}")):
        fvt_out = dataset_path / "2-av-pred_FVT" / (raw_file.stem + ".png")
        automorph_out = dataset_path / "2-av-pred_Automorph" / (raw_file.stem + ".png")
        vascx_out = dataset_path / "2-av-pred_VascX" / (raw_file.stem + ".png")

        if fvt_out.exists() and automorph_out.exists() and vascx_out.exists():
            continue

        fundus = FundusData(image=raw_file)
        fundus_cropped, roi = fundus.crop_to_roi(return_roi=True)

        if not fvt_out.exists():
            segment_av(fundus_cropped)
            fundus.update(av=fundus_cropped.av, roi=roi).write_image(av=fvt_out)

        if not automorph_out.exists():
            automorph_segment_av(fundus_cropped)
            fundus.update(av=fundus_cropped.av, roi=roi).write_image(av=automorph_out)

        if not vascx_out.exists():
            vascx_segment_av(fundus_cropped)
            fundus.update(av=fundus_cropped.av, roi=roi).write_image(av=vascx_out)

50it [00:00, 26465.83it/s]
200it [00:00, 33543.70it/s]
100it [00:00, 31216.91it/s]
44it [00:00, 28192.69it/s]
22it [00:00, 31633.42it/s]
15it [00:00, 29181.15it/s]
20it [00:00, 28397.45it/s]
20it [00:00, 29341.06it/s]


In [11]:
RAW = [path / "1-images" for path in DATASETS_PATH.values()]
TOPO = [path / "3-topo" for path in DATASETS_PATH.values()]
AV = [
    {
        "fvt": path / "2-av-pred_FVT",
        "automorph": path / "2-av-pred_Automorph",
        "gt": path / "2-av",
        "vascx": path / "2-av-pred_VascX",
    }
    for path in DATASETS_PATH.values()
]

dataset = BranchDigraphDataset.load_from_dirs(
    RAW,
    TOPO,
    AV,
    dataset_name=list(DATASETS_PATH.keys()),
    resize_to=1024,
    output_dir="tmp/ALL_DATA",
    n_workers=0,
    mask_optic_disc=True,
)

Found 373 branch digraphs...


Processing...
Done!


In [15]:
dataset.select_dataset("MAPLES-DR").jppype_show(10, augment=False)[0]

GridBox(children=(HTML(value='<h3 style="text-align: center;">20051110_35926_0400_PP/automorph</h3>'), HTML(va…

## Define dataset splits


In [ ]:
samples_by_dataset: dict[str, list[SampleInfo]] = {}
for sample in dataset.samples_info:
    samples_by_dataset.setdefault(sample.dataset, []).append(sample)

samples_stratification: dict[str, list[SampleInfo]] = {}


**Fundus AV**: stratify by pathology


In [ ]:
for sample in samples_by_dataset["FundusAV"]:
    samples_stratification.setdefault(f"FundusAV-{sample.name[-1]}", []).append(sample)

**LES-AV**, **INSPIRE** and **GAVE-train**: no specific stratification


In [ ]:
samples_stratification["LES-AV"] = samples_by_dataset["LES-AV"]
samples_stratification["INSPIRE"] = samples_by_dataset["INSPIRE"]
samples_stratification["GAVE-train"] = samples_by_dataset["GAVE-train"]
samples_stratification["HRF"] = samples_by_dataset["HRF"]

In [ ]:
for samples in samples_stratification.values():
    N = len(samples)
    train_end, val_end = int(N * 0.7), int(N * 0.85)
    for sample in samples[:train_end]:
        sample.dataset_type = "train"
    for sample in samples[train_end:val_end]:
        sample.dataset_type = "validation"
    for sample in samples[val_end:]:
        sample.dataset_type = "test"

Use existing splits for **DRIVE** and **MAPLES-DR**


In [ ]:
for sample in samples_by_dataset["DRIVE_train"]:
    sample.dataset_type = "validation" if sample.name.startswith(("31", "33", "35", "28")) else "train"
for sample in samples_by_dataset["DRIVE_test"]:
    sample.dataset_type = "test"

In [ ]:
import maples_dr

maples_dr_test_samples_name = {s.name for s in maples_dr.load_test_set()}
maples_dr_test_samples: list[SampleInfo] = []
for sample in samples_by_dataset["MAPLES-DR"]:
    if sample.name in maples_dr_test_samples_name:
        maples_dr_test_samples.append(sample)
    else:
        sample.dataset_type = "train"
N_test = len(maples_dr_test_samples)
for sample in maples_dr_test_samples[: N_test // 2]:
    sample.dataset_type = "validation"
for sample in maples_dr_test_samples[N_test // 2 :]:
    sample.dataset_type = "test"

Check that all samples have a dataset type assigned and save the splits


In [ ]:
samples_without_type = [
    sample for sample in dataset.samples_info if sample.dataset_type not in ("train", "validation", "test")
]
assert not samples_without_type, f"Samples without dataset type: {[s.name for s in samples_without_type]}"

dataset.save_manifest()

In [ ]:
train_set, val_set, test_set = dataset.split_sets()


def count_dataset(dataset):
    counts = {}
    for sample in dataset.samples_info:
        counts[sample.dataset] = counts.get(sample.dataset, 0) + 1
    return counts


df = pd.DataFrame(
    {"train": count_dataset(train_set), "validation": count_dataset(val_set), "test": count_dataset(test_set)}
).T
df["TOTAL"] = df.sum(axis=1)
df

## Bundle dataset


In [ ]:
dataset.bundle("ALL_DATA_bundle.tar.gz", overwrite=True)

In [ ]:
bundled_dataset = BranchDigraphDataset("ALL_DATA_bundle.tar.gz")

In [ ]:
train_set, val_set, test_set = bundled_dataset.split_sets()


def count_dataset(dataset):
    counts = {}
    for sample in dataset.samples_info:
        counts[sample.dataset] = counts.get(sample.dataset, 0) + 1
    return counts


df = pd.DataFrame(
    {"train": count_dataset(train_set), "validation": count_dataset(val_set), "test": count_dataset(test_set)}
).T
df["TOTAL"] = df.sum(axis=1)
df

## Test


In [ ]:
m, sample, sample_data = val_set.jppype_show(51, augment=False, version="fvt")

m